In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Notebook 1 — Bronze Layer
-- MAGIC **Retail Pharmacy Chain — Shampoo Promotion Analysis**
-- MAGIC
-- MAGIC Ingest all 5 RAW CSV files into Delta Lake exactly as-is.
-- MAGIC No cleaning. Audit metadata only.
-- MAGIC
-- MAGIC ```
-- MAGIC RAW_Product.csv    →  bronze_product    (2 products + dirty dups)
-- MAGIC RAW_Store.csv      →  bronze_store      (1,300 Canadian pharmacy stores)
-- MAGIC RAW_Date.csv       →  bronze_date       (Year × Week calendar)
-- MAGIC RAW_Promotion.csv  →  bronze_promotion  (price/flyer promotion types)
-- MAGIC RAW_Sales.csv      →  bronze_sales      (store × product × week — fact)
-- MAGIC ```
-- MAGIC
-- MAGIC > **Grain**: 1 row = 1 store × 1 product × 1 week.
-- MAGIC > `SUM(Units, SalesAmt, GrossMargin, Transactions)` grouped by
-- MAGIC > `Year, WeekNumber, Product` reproduces the original chain-level dataset.

-- COMMAND ----------
-- MAGIC %md ## 0. Configuration

-- COMMAND ----------

CREATE SCHEMA IF NOT EXISTS workspace.promotion_sql;
USE workspace.promotion_sql;

-- COMMAND ----------
-- MAGIC %md ## 1. Bronze Tables — Load CSV files

-- COMMAND ----------
-- MAGIC %md ### 1.1 Product

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.bronze_product
USING DELTA AS
SELECT 
    *,
    current_timestamp() AS _ingest_time,
    _metadata.file_path AS _source_file,
    current_date() AS _batch_date
FROM read_files(
    '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Product.csv',
    format => 'csv',
    header => true,
    inferSchema => false,
    multiLine => true,
    escape => '"'
);

-- COMMAND ----------

SELECT * FROM bronze_product;

-- COMMAND ----------
-- MAGIC %md ### 1.2 Store

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.bronze_store
USING DELTA AS
SELECT 
    *,
    current_timestamp() AS _ingest_time,
    _metadata.file_path AS _source_file,
    current_date() AS _batch_date
FROM read_files(
    '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Store.csv',
    format => 'csv',
    header => true,
    inferSchema => false,
    multiLine => true,
    escape => '"'
);

-- COMMAND ----------

SELECT * FROM bronze_store LIMIT 5;

-- COMMAND ----------
-- MAGIC %md ### 1.3 Date

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.bronze_date
USING DELTA AS
SELECT 
    *,
    current_timestamp() AS _ingest_time,
    _metadata.file_path AS _source_file,
    current_date() AS _batch_date
FROM read_files(
    '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Date.csv',
    format => 'csv',
    header => true,
    inferSchema => false,
    multiLine => true,
    escape => '"'
);

-- COMMAND ----------

SELECT * FROM bronze_date LIMIT 8;

-- COMMAND ----------
-- MAGIC %md ### 1.4 Promotion

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.bronze_promotion
USING DELTA AS
SELECT 
    *,
    current_timestamp() AS _ingest_time,
    _metadata.file_path AS _source_file,
    current_date() AS _batch_date
FROM read_files(
    '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Promotion.csv',
    format => 'csv',
    header => true,
    inferSchema => false,
    multiLine => true,
    escape => '"'
);

-- COMMAND ----------

SELECT * FROM bronze_promotion;

-- COMMAND ----------
-- MAGIC %md ## 2. Fact table — Sales (store × product × week)

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.bronze_sales
USING DELTA AS
SELECT 
    *,
    current_timestamp() AS _ingest_time,
    _metadata.file_path AS _source_file,
    current_date() AS _batch_date
FROM read_files(
    '/Volumes/workspace/default/course_data/Promotion_raw_data/RAW_Sales.csv',
    format => 'csv',
    header => true,
    inferSchema => false,
    multiLine => true,
    escape => '"'
);

-- COMMAND ----------

SELECT * FROM bronze_sales LIMIT 5;

-- COMMAND ----------
-- MAGIC %md ## 3. Row count summary

-- COMMAND ----------

SELECT 'product'   AS table_name, COUNT(*) AS rows FROM bronze_product
UNION ALL SELECT 'store',         COUNT(*)         FROM bronze_store
UNION ALL SELECT 'date',          COUNT(*)         FROM bronze_date
UNION ALL SELECT 'promotion',     COUNT(*)         FROM bronze_promotion
UNION ALL SELECT 'sales',         COUNT(*)         FROM bronze_sales
ORDER BY table_name;

-- COMMAND ----------
-- MAGIC %md ## 4. Spot the data problems — do NOT fix here, that is Silver's job

-- COMMAND ----------

-- Product: mixed Category/Supplier casing, duplicate rows
SELECT * FROM bronze_product;

-- COMMAND ----------

-- Promotion: Discount in 3 formats (0.10  /  "10%"  /  10.0),
--            mixed OnFlyer casing, mixed PromotionType casing, NULL DiscountTier
SELECT * FROM bronze_promotion;

-- COMMAND ----------

-- Date: FiscalYear in 3 formats (FY2021 / 2021 / FY-2021),
--       mixed Month casing, NULL Quarter
SELECT * FROM bronze_date LIMIT 15;

-- COMMAND ----------

-- Sales: mixed Product casing, mixed OnFlyer casing,
--        Discount in 3 formats, NULL Price, 50 duplicate rows
SELECT Year, WeekNumber, StoreID, Product, Price, OnFlyer, Discount,
       Units, SalesAmt, GrossMargin, Transactions
FROM bronze_sales
LIMIT 15;

-- COMMAND ----------

-- How many distinct Discount formats appear in sales?
SELECT Discount, COUNT(*) AS rows
FROM bronze_sales
GROUP BY Discount
ORDER BY rows DESC;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ## ✅ Bronze Complete
-- MAGIC All 5 tables landed unchanged. Proceed to **Notebook 2 — Silver**.

In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Notebook 2 — Silver Layer: Cleaning + Data Modeling
-- MAGIC
-- MAGIC **Steps:**
-- MAGIC 1. Clean each Bronze dimension table
-- MAGIC 2. Generate Surrogate Keys
-- MAGIC 3. Join fact_sales to all dimensions — swap natural keys for surrogate keys
-- MAGIC
-- MAGIC **Star Schema output:**
-- MAGIC ```
-- MAGIC dim_product    (ProductKey)    ─┐
-- MAGIC dim_store      (StoreKey)      ─┤
-- MAGIC dim_date       (DateKey)       ─┼──► fact_sales
-- MAGIC dim_promotion  (PromotionKey)  ─┘
-- MAGIC ```
-- MAGIC
-- MAGIC **Key join challenge:**
-- MAGIC `RAW_Sales` has no `PromotionName` column.
-- MAGIC The promotion must be derived from `OnFlyer + Discount` — after cleaning both —
-- MAGIC then joined to `dim_promotion` on `PromotionName`.

-- COMMAND ----------
-- MAGIC %md ## 0. Configuration

-- COMMAND ----------

USE workspace.promotion_sql;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 1. dim_product
-- MAGIC **Natural key**: `Product` (name)
-- MAGIC **Cleaning**: title-case all string cols, dedup
-- MAGIC **Key**: `ProductKey` = row_number ordered by Product

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.silver_dim_product
USING DELTA AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY Product) AS ProductKey,
    Product,
    Brand,
    Category,
    Size,
    Supplier,
    UnitCost
FROM (
    SELECT DISTINCT
        INITCAP(TRIM(Product)) AS Product,
        INITCAP(TRIM(Brand)) AS Brand,
        INITCAP(TRIM(Category)) AS Category,
        TRIM(Size) AS Size,
        INITCAP(TRIM(Supplier)) AS Supplier,
        TRY_CAST(UnitCost AS DOUBLE) AS UnitCost
    FROM workspace.promotion_sql.bronze_product
);

-- COMMAND ----------

SELECT * FROM silver_dim_product;

-- COMMAND ----------

SELECT COUNT(*) AS row_count FROM silver_dim_product;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 2. dim_store
-- MAGIC **Natural key**: `StoreID`
-- MAGIC **Cleaning**: title-case City/Province, upper-case ProvinceAbbrev, dedup
-- MAGIC **Key**: `StoreKey` = row_number ordered by StoreID

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.silver_dim_store
USING DELTA AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY StoreID) AS StoreKey,
    StoreID,
    StoreName,
    City,
    Province,
    ProvinceAbbrev,
    Country
FROM (
    SELECT DISTINCT
        TRIM(StoreID) AS StoreID,
        INITCAP(TRIM(StoreName)) AS StoreName,
        INITCAP(TRIM(City)) AS City,
        INITCAP(TRIM(Province)) AS Province,
        UPPER(TRIM(ProvinceAbbrev)) AS ProvinceAbbrev,
        INITCAP(TRIM(Country)) AS Country
    FROM workspace.promotion_sql.bronze_store
);

-- COMMAND ----------

SELECT Province, COUNT(*) as count 
FROM silver_dim_store 
GROUP BY Province 
ORDER BY count DESC;

-- COMMAND ----------

SELECT COUNT(*) AS row_count FROM silver_dim_store;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 3. dim_date
-- MAGIC **Natural key**: `Year` + `WeekNumber`
-- MAGIC **Cleaning**: standardise FiscalYear → "FY2021", title-case Month, fill NULL Quarter
-- MAGIC **Key**: `DateKey` = Year × 100 + WeekNumber  →  202101

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.silver_dim_date
USING DELTA AS
SELECT 
    CAST(Year * 100 + WeekNumber AS INT) AS DateKey,
    Year,
    WeekNumber,
    WeekStartDate,
    Month,
    MonthNumber,
    Quarter,
    FiscalYear
FROM (
    SELECT DISTINCT
        CAST(Year AS INT) AS Year,
        CAST(WeekNumber AS INT) AS WeekNumber,
        WeekStartDate,
        INITCAP(TRIM(Month)) AS Month,
        CAST(MonthNumber AS INT) AS MonthNumber,
        -- Standardise FiscalYear: "FY2021" / "2021" / "FY-2021" → "FY2021"
        CONCAT('FY', REGEXP_EXTRACT(FiscalYear, '(\\d{4})', 1)) AS FiscalYear,
        -- Fill NULL Quarter from MonthNumber
        -- Retail quarters: Q1=Feb-Apr, Q2=May-Jul, Q3=Aug-Oct, Q4=Nov-Jan
        CASE 
            WHEN Quarter IS NOT NULL THEN Quarter
            WHEN MonthNumber IN (2,3,4) THEN 'Q1'
            WHEN MonthNumber IN (5,6,7) THEN 'Q2'
            WHEN MonthNumber IN (8,9,10) THEN 'Q3'
            ELSE 'Q4'
        END AS Quarter
    FROM workspace.promotion_sql.bronze_date
);

-- COMMAND ----------

SELECT * FROM silver_dim_date ORDER BY DateKey LIMIT 10;

-- COMMAND ----------

SELECT COUNT(*) AS row_count FROM silver_dim_date;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 4. dim_promotion
-- MAGIC **Natural key**: `PromotionName`
-- MAGIC **Cleaning**: fix Discount format, standardise casing, fill NULL DiscountTier
-- MAGIC **Key**: `PromotionKey` = row_number ordered by PromotionName
-- MAGIC
-- MAGIC > The cleaned `PromotionName` + `OnFlyer` + `Discount` will be used
-- MAGIC > in the fact join to identify which promotion applied to each sales row.

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.silver_dim_promotion
USING DELTA AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY PromotionName) AS PromotionKey,
    PromotionName,
    OnFlyer,
    Discount,
    PromotionType,
    DiscountTier
FROM (
    SELECT DISTINCT
        TRIM(PromotionName) AS PromotionName,
        INITCAP(TRIM(OnFlyer)) AS OnFlyer,
        -- Handle 3 Discount formats: "10%" → 0.10 | 10.0 → 0.10 | 0.10 → 0.10
        CASE 
            WHEN TRIM(Discount) LIKE '%\\%' THEN 
                TRY_CAST(REGEXP_EXTRACT(TRIM(Discount), '([\\d\\.]+)', 1) AS DOUBLE) / 100
            WHEN TRY_CAST(Discount AS DOUBLE) > 1 THEN 
                TRY_CAST(Discount AS DOUBLE) / 100
            ELSE 
                TRY_CAST(Discount AS DOUBLE)
        END AS Discount,
        INITCAP(TRIM(PromotionType)) AS PromotionType,
        -- Fill NULL DiscountTier from Discount value
        CASE 
            WHEN TRIM(DiscountTier) IS NOT NULL THEN INITCAP(TRIM(DiscountTier))
            WHEN CAST(CASE 
                    WHEN TRIM(Discount) LIKE '%\\%' THEN 
                        TRY_CAST(REGEXP_EXTRACT(TRIM(Discount), '([\\d\\.]+)', 1) AS DOUBLE) / 100
                    WHEN TRY_CAST(Discount AS DOUBLE) > 1 THEN 
                        TRY_CAST(Discount AS DOUBLE) / 100
                    ELSE 
                        TRY_CAST(Discount AS DOUBLE)
                END AS DOUBLE) >= 0.30 THEN 'Deep'
            WHEN CAST(CASE 
                    WHEN TRIM(Discount) LIKE '%\\%' THEN 
                        TRY_CAST(REGEXP_EXTRACT(TRIM(Discount), '([\\d\\.]+)', 1) AS DOUBLE) / 100
                    WHEN TRY_CAST(Discount AS DOUBLE) > 1 THEN 
                        TRY_CAST(Discount AS DOUBLE) / 100
                    ELSE 
                        TRY_CAST(Discount AS DOUBLE)
                END AS DOUBLE) > 0.00 THEN 'Mid'
            ELSE 'None'
        END AS DiscountTier
    FROM workspace.promotion_sql.bronze_promotion
);

-- COMMAND ----------

SELECT * FROM silver_dim_promotion ORDER BY Discount, OnFlyer;

-- COMMAND ----------

SELECT COUNT(*) AS row_count FROM silver_dim_promotion;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 5. fact_sales
-- MAGIC
-- MAGIC This is the core modeling step.
-- MAGIC `RAW_Sales` has no surrogate keys and no `PromotionName`.
-- MAGIC We must:
-- MAGIC 1. Clean `Product`, `OnFlyer`, `Discount`, cast numeric columns, fill NULL `Price`
-- MAGIC 2. **Derive `PromotionName`** from cleaned `OnFlyer + Discount` (same logic as dim_promotion)
-- MAGIC 3. Join all 4 dimensions to get surrogate keys
-- MAGIC 4. Recalculate `SalesAmt` and `GrossMargin` for rows where `Price` was NULL
-- MAGIC
-- MAGIC **Join chain:**
-- MAGIC ```
-- MAGIC RAW_Sales
-- MAGIC   + dim_product   on  Product                     → ProductKey
-- MAGIC   + dim_store     on  StoreID                     → StoreKey
-- MAGIC   + dim_date      on  Year + WeekNumber           → DateKey
-- MAGIC   + dim_promotion on  derived PromotionName       → PromotionKey
-- MAGIC ```

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.silver_fact_sales
USING DELTA AS
WITH cleaned_sales AS (
    SELECT DISTINCT
        CAST(Year AS INT) AS Year,
        CAST(WeekNumber AS INT) AS WeekNumber,
        TRIM(StoreID) AS StoreID,
        INITCAP(TRIM(Product)) AS Product,
        TRY_CAST(Price AS DOUBLE) AS Price,
        INITCAP(TRIM(OnFlyer)) AS OnFlyer,
        -- Handle 3 Discount formats
        CASE 
            WHEN TRIM(Discount) LIKE '%\\%' THEN 
                TRY_CAST(REGEXP_EXTRACT(TRIM(Discount), '([\\d\\.]+)', 1) AS DOUBLE) / 100
            WHEN TRY_CAST(Discount AS DOUBLE) > 1 THEN 
                TRY_CAST(Discount AS DOUBLE) / 100
            ELSE 
                TRY_CAST(Discount AS DOUBLE)
        END AS Discount,
        CAST(Units AS INT) AS Units,
        TRY_CAST(SalesAmt AS DOUBLE) AS SalesAmt,
        TRY_CAST(GrossMargin AS DOUBLE) AS GrossMargin,
        CAST(Transactions AS INT) AS Transactions
    FROM workspace.promotion_sql.bronze_sales
),
with_promotion_name AS (
    SELECT 
        *,
        -- Derive PromotionName (same formula used to build dim_promotion)
        CASE 
            WHEN Discount = 0 THEN 'No Promotion'
            WHEN OnFlyer = 'Yes' THEN CONCAT(CAST(CAST(Discount * 100 AS INT) AS STRING), '% Off + Flyer')
            ELSE CONCAT(CAST(CAST(Discount * 100 AS INT) AS STRING), '% Off')
        END AS PromotionName
    FROM cleaned_sales
),
with_keys AS (
    SELECT 
        p.ProductKey,
        s.StoreKey,
        d.DateKey,
        pr.PromotionKey,
        cs.Year,
        cs.WeekNumber,
        cs.Price,
        cs.Discount,
        cs.Units,
        cs.SalesAmt,
        cs.GrossMargin,
        cs.Transactions,
        p.UnitCost
    FROM with_promotion_name cs
    LEFT JOIN workspace.promotion_sql.silver_dim_product p ON cs.Product = p.Product
    LEFT JOIN workspace.promotion_sql.silver_dim_store s ON cs.StoreID = s.StoreID
    LEFT JOIN workspace.promotion_sql.silver_dim_date d ON cs.Year = d.Year AND cs.WeekNumber = d.WeekNumber
    LEFT JOIN workspace.promotion_sql.silver_dim_promotion pr ON cs.PromotionName = pr.PromotionName
)
SELECT 
    ProductKey,
    StoreKey,
    DateKey,
    PromotionKey,
    Year,
    WeekNumber,
    -- Fix NULL Price rows: recalculate from SalesAmt / Units
    CASE 
        WHEN Price IS NULL THEN ROUND(SalesAmt / Units, 2)
        ELSE Price
    END AS ActualPrice,
    Discount AS DiscountPct,
    Units AS UnitsSold,
    SalesAmt,
    GrossMargin,
    Transactions,
    -- Flag below-cost promotions (negative GrossMargin — loss leaders)
    CASE WHEN GrossMargin < 0 THEN 1 ELSE 0 END AS IsBelowCost
FROM with_keys;

-- COMMAND ----------

-- Join quality check
SELECT 'Total rows' AS metric, COUNT(*) AS value FROM silver_fact_sales
UNION ALL SELECT 'NULL ProductKey', COUNT(*) FROM silver_fact_sales WHERE ProductKey IS NULL
UNION ALL SELECT 'NULL StoreKey', COUNT(*) FROM silver_fact_sales WHERE StoreKey IS NULL
UNION ALL SELECT 'NULL DateKey', COUNT(*) FROM silver_fact_sales WHERE DateKey IS NULL
UNION ALL SELECT 'NULL PromotionKey', COUNT(*) FROM silver_fact_sales WHERE PromotionKey IS NULL
UNION ALL SELECT 'Below-cost rows', COUNT(*) FROM silver_fact_sales WHERE IsBelowCost = 1;

-- COMMAND ----------
-- MAGIC %md ## 6. Verify: aggregate back to chain level

-- COMMAND ----------

-- Re-aggregate store rows → should match original chain-level numbers
SELECT
    d.Year,
    d.WeekNumber,
    p.Product,
    pr.PromotionName,
    ROUND(AVG(f.ActualPrice), 2) AS Price,
    SUM(f.UnitsSold) AS TotalUnits,
    ROUND(SUM(f.SalesAmt), 2) AS TotalSalesAmt,
    ROUND(SUM(f.GrossMargin), 2) AS TotalGrossMargin,
    SUM(f.Transactions) AS TotalTransactions
FROM silver_fact_sales f
JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
JOIN silver_dim_date d ON f.DateKey = d.DateKey
JOIN silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
GROUP BY d.Year, d.WeekNumber, p.Product, pr.PromotionName
ORDER BY d.Year, d.WeekNumber, p.Product
LIMIT 20;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ## ✅ Silver Complete
-- MAGIC Star schema ready for analysis.

In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Notebook 3 — Gold Layer: Promotion Analysis
-- MAGIC
-- MAGIC Each Gold table directly answers one or more project questions:
-- MAGIC
-- MAGIC | Gold Table | Project Question |
-- MAGIC |------------|------------------|
-- MAGIC | `gold_price_elasticity`   | Q1/Q2: Which price maximises units/margin? |
-- MAGIC | `gold_promotion_uplift`   | Q5/Q6/Q7: Discount impact vs baseline |
-- MAGIC | `gold_weekly_trend`       | Q3: Is shampoo seasonal? |
-- MAGIC | `gold_province_summary`   | Chain-level + province breakdown |
-- MAGIC | `gold_loss_leader`        | Q9/Q10: Is Aussie @$2.49 an effective loss leader? |

-- COMMAND ----------
-- MAGIC %md ## 0. Configuration

-- COMMAND ----------

USE workspace.promotion_sql;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 1. gold_price_elasticity
-- MAGIC **Answers Q1 & Q2**: Which price point maximises units sold? Which maximises gross margin?
-- MAGIC
-- MAGIC Aggregates all store-weeks at each price point to get chain totals,
-- MAGIC then calculates average weekly performance per price.

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.gold_price_elasticity
USING DELTA AS
WITH chain_weekly AS (
    SELECT 
        f.ProductKey,
        p.Product,
        p.UnitCost,
        f.PromotionKey,
        pr.PromotionName,
        pr.OnFlyer,
        pr.DiscountTier,
        f.ActualPrice,
        f.DiscountPct,
        f.Year,
        f.WeekNumber,
        SUM(f.UnitsSold) AS ChainUnits,
        SUM(f.SalesAmt) AS ChainSalesAmt,
        SUM(f.GrossMargin) AS ChainGrossMargin,
        SUM(f.Transactions) AS ChainTransactions,
        SUM(f.IsBelowCost) AS StoresBelowCost,
        COUNT(f.StoreKey) AS StoresActive
    FROM workspace.promotion_sql.silver_fact_sales f
    JOIN workspace.promotion_sql.silver_dim_product p ON f.ProductKey = p.ProductKey
    JOIN workspace.promotion_sql.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
    GROUP BY f.ProductKey, p.Product, p.UnitCost, f.PromotionKey, pr.PromotionName,
             pr.OnFlyer, pr.DiscountTier, f.ActualPrice, f.DiscountPct, f.Year, f.WeekNumber
),
price_aggregates AS (
    SELECT 
        ProductKey,
        Product,
        UnitCost,
        PromotionName,
        OnFlyer,
        DiscountTier,
        ActualPrice,
        DiscountPct,
        COUNT(WeekNumber) AS WeeksAtThisPrice,
        ROUND(AVG(ChainUnits), 0) AS AvgWeeklyUnits,
        ROUND(SUM(ChainUnits), 0) AS TotalUnits,
        ROUND(AVG(ChainSalesAmt), 2) AS AvgWeeklySales,
        ROUND(SUM(ChainSalesAmt), 2) AS TotalSales,
        ROUND(AVG(ChainGrossMargin), 2) AS AvgWeeklyMargin,
        ROUND(SUM(ChainGrossMargin), 2) AS TotalMargin,
        ROUND(AVG(ChainTransactions), 0) AS AvgWeeklyTransactions,
        ROUND(AVG(StoresActive), 0) AS AvgStoresActive
    FROM chain_weekly
    GROUP BY ProductKey, Product, UnitCost, PromotionName, OnFlyer, 
             DiscountTier, ActualPrice, DiscountPct
)
SELECT 
    ProductKey,
    Product,
    UnitCost,
    PromotionName,
    OnFlyer,
    DiscountTier,
    ActualPrice,
    DiscountPct,
    WeeksAtThisPrice,
    AvgWeeklyUnits,
    TotalUnits,
    AvgWeeklySales,
    TotalSales,
    AvgWeeklyMargin,
    TotalMargin,
    AvgWeeklyTransactions,
    AvgStoresActive,
    ROUND(TotalMargin / TotalSales * 100, 1) AS GrossMarginPct,
    ROUND(AvgWeeklyMargin / AvgWeeklyUnits, 4) AS MarginPerUnit,
    RANK() OVER (PARTITION BY Product ORDER BY AvgWeeklyUnits DESC) AS RankByUnits,
    RANK() OVER (PARTITION BY Product ORDER BY AvgWeeklyMargin DESC) AS RankByMargin,
    CURRENT_TIMESTAMP() AS _gold_ts
FROM price_aggregates
ORDER BY Product, ActualPrice;

-- COMMAND ----------

SELECT Product, ActualPrice, OnFlyer, DiscountPct, WeeksAtThisPrice,
       AvgWeeklyUnits, AvgWeeklyMargin, GrossMarginPct, RankByUnits, RankByMargin
FROM gold_price_elasticity
ORDER BY Product, ActualPrice
LIMIT 25;

-- COMMAND ----------

SELECT COUNT(*) AS row_count FROM gold_price_elasticity;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 2. gold_promotion_uplift
-- MAGIC **Answers Q5, Q6, Q7**: Uplift of each promotion vs regular-price baseline.
-- MAGIC Includes the specific 25% and 60% scenarios (Q5/Q6).
-- MAGIC
-- MAGIC > Baseline = average chain weekly units/margin when Discount = 0.

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.gold_promotion_uplift
USING DELTA AS
WITH baseline AS (
    SELECT 
        f.ProductKey,
        ROUND(AVG(week_agg.WeekUnits), 0) AS BaselineUnits,
        ROUND(AVG(week_agg.WeekSales), 2) AS BaselineSales,
        ROUND(AVG(week_agg.WeekMargin), 2) AS BaselineMargin
    FROM workspace.promotion_sql.silver_fact_sales f
    JOIN workspace.promotion_sql.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
    JOIN workspace.promotion_sql.silver_dim_date d ON f.DateKey = d.DateKey
    JOIN (
        SELECT 
            ProductKey,
            Year,
            WeekNumber,
            SUM(UnitsSold) AS WeekUnits,
            SUM(SalesAmt) AS WeekSales,
            SUM(GrossMargin) AS WeekMargin
        FROM workspace.promotion_sql.silver_fact_sales f2
        JOIN workspace.promotion_sql.silver_dim_promotion pr2 ON f2.PromotionKey = pr2.PromotionKey
        WHERE pr2.Discount = 0
        GROUP BY ProductKey, Year, WeekNumber
    ) week_agg ON f.ProductKey = week_agg.ProductKey 
                  AND f.Year = week_agg.Year 
                  AND f.WeekNumber = week_agg.WeekNumber
    WHERE pr.Discount = 0
    GROUP BY f.ProductKey
),
promo_performance AS (
    SELECT 
        f.ProductKey,
        p.Product,
        f.PromotionKey,
        pr.PromotionName,
        pr.OnFlyer,
        pr.Discount,
        pr.DiscountTier,
        COUNT(DISTINCT CONCAT(CAST(f.Year AS STRING), '-', CAST(f.WeekNumber AS STRING))) AS WeeksRan,
        ROUND(AVG(week_agg.WeekUnits), 0) AS AvgWeeklyUnits,
        ROUND(AVG(week_agg.WeekSales), 2) AS AvgWeeklySales,
        ROUND(AVG(week_agg.WeekMargin), 2) AS AvgWeeklyMargin,
        ROUND(SUM(week_agg.WeekMargin), 2) AS TotalMargin,
        SUM(week_agg.StoresBelowCost) AS BelowCostInstances
    FROM workspace.promotion_sql.silver_fact_sales f
    JOIN workspace.promotion_sql.silver_dim_product p ON f.ProductKey = p.ProductKey
    JOIN workspace.promotion_sql.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
    JOIN workspace.promotion_sql.silver_dim_date d ON f.DateKey = d.DateKey
    JOIN (
        SELECT 
            ProductKey,
            PromotionKey,
            Year,
            WeekNumber,
            SUM(UnitsSold) AS WeekUnits,
            SUM(SalesAmt) AS WeekSales,
            SUM(GrossMargin) AS WeekMargin,
            SUM(IsBelowCost) AS StoresBelowCost
        FROM workspace.promotion_sql.silver_fact_sales
        GROUP BY ProductKey, PromotionKey, Year, WeekNumber
    ) week_agg ON f.ProductKey = week_agg.ProductKey 
                  AND f.PromotionKey = week_agg.PromotionKey
                  AND f.Year = week_agg.Year 
                  AND f.WeekNumber = week_agg.WeekNumber
    GROUP BY f.ProductKey, p.Product, f.PromotionKey, pr.PromotionName,
             pr.OnFlyer, pr.Discount, pr.DiscountTier
)
SELECT 
    pp.ProductKey,
    pp.Product,
    pp.PromotionKey,
    pp.PromotionName,
    pp.OnFlyer,
    pp.Discount,
    pp.DiscountTier,
    pp.WeeksRan,
    pp.AvgWeeklyUnits,
    pp.AvgWeeklySales,
    pp.AvgWeeklyMargin,
    pp.TotalMargin,
    pp.BelowCostInstances,
    b.BaselineUnits,
    b.BaselineSales,
    b.BaselineMargin,
    ROUND((pp.AvgWeeklyUnits - b.BaselineUnits) / b.BaselineUnits * 100, 1) AS UnitUpliftPct,
    ROUND((pp.AvgWeeklySales - b.BaselineSales) / b.BaselineSales * 100, 1) AS SalesUpliftPct,
    ROUND((pp.AvgWeeklyMargin - b.BaselineMargin) / b.BaselineMargin * 100, 1) AS MarginUpliftPct,
    ROUND(pp.AvgWeeklyUnits - b.BaselineUnits, 0) AS IncrementalUnits,
    ROUND(pp.AvgWeeklyMargin - b.BaselineMargin, 2) AS IncrementalMargin,
    CURRENT_TIMESTAMP() AS _gold_ts
FROM promo_performance pp
LEFT JOIN baseline b ON pp.ProductKey = b.ProductKey
ORDER BY pp.Product, pp.Discount;

-- COMMAND ----------

SELECT Product, PromotionName, OnFlyer, Discount,
       AvgWeeklyUnits, BaselineUnits, UnitUpliftPct,
       AvgWeeklyMargin, BaselineMargin, MarginUpliftPct,
       BelowCostInstances
FROM gold_promotion_uplift
ORDER BY Product, Discount
LIMIT 25;

-- COMMAND ----------

SELECT COUNT(*) AS row_count FROM gold_promotion_uplift;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 3. gold_weekly_trend
-- MAGIC **Answers Q3**: Is shampoo seasonal?
-- MAGIC Weekly chain-level units/sales/margin with 4-week moving average to smooth noise.

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.gold_weekly_trend
USING DELTA AS
WITH chain_weekly AS (
    SELECT 
        p.Product,
        d.Year AS DateYear,
        d.WeekNumber AS DateWeekNumber,
        d.Month,
        d.MonthNumber,
        d.Quarter,
        d.FiscalYear,
        d.WeekStartDate,
        pr.PromotionName,
        pr.OnFlyer,
        pr.Discount,
        SUM(f.UnitsSold) AS ChainUnits,
        ROUND(SUM(f.SalesAmt), 2) AS ChainSalesAmt,
        ROUND(SUM(f.GrossMargin), 2) AS ChainGrossMargin,
        SUM(f.Transactions) AS ChainTransactions,
        SUM(f.IsBelowCost) AS StoresBelowCost,
        COUNT(f.StoreKey) AS StoreCount
    FROM workspace.promotion_sql.silver_fact_sales f
    JOIN workspace.promotion_sql.silver_dim_product p ON f.ProductKey = p.ProductKey
    JOIN workspace.promotion_sql.silver_dim_date d ON f.DateKey = d.DateKey
    JOIN workspace.promotion_sql.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
    GROUP BY p.Product, d.Year, d.WeekNumber, d.Month, d.MonthNumber,
             d.Quarter, d.FiscalYear, d.WeekStartDate, pr.PromotionName, pr.OnFlyer, pr.Discount
)
SELECT 
    Product,
    DateYear,
    DateWeekNumber,
    Month,
    MonthNumber,
    Quarter,
    FiscalYear,
    WeekStartDate,
    PromotionName,
    OnFlyer,
    Discount,
    ChainUnits,
    ChainSalesAmt,
    ChainGrossMargin,
    ChainTransactions,
    StoresBelowCost,
    StoreCount,
    CAST(DateYear * 100 + DateWeekNumber AS INT) AS DateKey,
    ROUND(
        AVG(ChainUnits) OVER (
            PARTITION BY Product
            ORDER BY CAST(DateYear * 100 + DateWeekNumber AS INT)
            ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
        ), 0
    ) AS RollingAvg4WkUnits,
    ROUND(
        (ChainUnits - LAG(ChainUnits, 1) OVER (PARTITION BY Product ORDER BY CAST(DateYear * 100 + DateWeekNumber AS INT)))
        / LAG(ChainUnits, 1) OVER (PARTITION BY Product ORDER BY CAST(DateYear * 100 + DateWeekNumber AS INT)) * 100, 1
    ) AS WoWChangePct,
    ROUND(ChainGrossMargin / ChainSalesAmt * 100, 1) AS GrossMarginPct,
    CASE WHEN Discount > 0 THEN 1 ELSE 0 END AS IsPromoWeek,
    CURRENT_TIMESTAMP() AS _gold_ts
FROM chain_weekly
ORDER BY Product, DateYear, DateWeekNumber;

-- COMMAND ----------

SELECT COUNT(*) AS row_count FROM gold_weekly_trend;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 4. gold_province_summary
-- MAGIC Chain-level + province breakdown for regional analysis.
-- MAGIC Supports: which provinces respond most to promotions?

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.gold_province_summary
USING DELTA AS
SELECT 
    p.Product,
    s.Province,
    s.ProvinceAbbrev,
    d.Year AS DateYear,
    d.WeekNumber AS DateWeekNumber,
    d.FiscalYear,
    d.Quarter,
    pr.PromotionName,
    pr.OnFlyer,
    pr.Discount,
    pr.DiscountTier,
    SUM(f.UnitsSold) AS TotalUnits,
    ROUND(SUM(f.SalesAmt), 2) AS TotalSales,
    ROUND(SUM(f.GrossMargin), 2) AS TotalMargin,
    SUM(f.Transactions) AS TotalTransactions,
    COUNT(f.StoreKey) AS StoreCount,
    SUM(f.IsBelowCost) AS BelowCostInstances,
    ROUND(SUM(f.GrossMargin) / SUM(f.SalesAmt) * 100, 1) AS GrossMarginPct,
    ROUND(SUM(f.UnitsSold) / COUNT(f.StoreKey), 1) AS UnitsPerStore,
    CURRENT_TIMESTAMP() AS _gold_ts
FROM workspace.promotion_sql.silver_fact_sales f
JOIN workspace.promotion_sql.silver_dim_product p ON f.ProductKey = p.ProductKey
JOIN workspace.promotion_sql.silver_dim_store s ON f.StoreKey = s.StoreKey
JOIN workspace.promotion_sql.silver_dim_date d ON f.DateKey = d.DateKey
JOIN workspace.promotion_sql.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
GROUP BY p.Product, s.Province, s.ProvinceAbbrev, d.Year, d.WeekNumber,
         d.FiscalYear, d.Quarter, pr.PromotionName, pr.OnFlyer, pr.Discount, pr.DiscountTier
ORDER BY p.Product, s.Province, d.Year, d.WeekNumber, d.Quarter;

-- COMMAND ----------

SELECT COUNT(*) AS row_count FROM gold_province_summary;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ---
-- MAGIC ## 5. gold_loss_leader
-- MAGIC **Answers Q9 & Q10**: Is Aussie @ $2.49 an effective loss leader?
-- MAGIC Compares the $2.49 promo week directly to all other Aussie price points.

-- COMMAND ----------

CREATE OR REPLACE TABLE workspace.promotion_sql.gold_loss_leader
USING DELTA AS
WITH chain_weekly AS (
    SELECT 
        f.ProductKey,
        p.Product,
        p.UnitCost,
        f.PromotionKey,
        pr.PromotionName,
        pr.OnFlyer,
        pr.Discount,
        pr.DiscountTier,
        d.Year AS DateYear,
        d.WeekNumber AS DateWeekNumber,
        f.ActualPrice,
        SUM(f.UnitsSold) AS ChainUnits,
        ROUND(SUM(f.SalesAmt), 2) AS ChainSales,
        ROUND(SUM(f.GrossMargin), 2) AS ChainMargin,
        SUM(f.Transactions) AS ChainTransactions,
        COUNT(f.StoreKey) AS StoreCount
    FROM workspace.promotion_sql.silver_fact_sales f
    JOIN workspace.promotion_sql.silver_dim_product p ON f.ProductKey = p.ProductKey
    JOIN workspace.promotion_sql.silver_dim_promotion pr ON f.PromotionKey = pr.PromotionKey
    JOIN workspace.promotion_sql.silver_dim_date d ON f.DateKey = d.DateKey
    GROUP BY f.ProductKey, p.Product, p.UnitCost, f.PromotionKey, pr.PromotionName,
             pr.OnFlyer, pr.Discount, pr.DiscountTier, d.Year, d.WeekNumber, f.ActualPrice
)
SELECT 
    Product,
    UnitCost,
    PromotionName,
    OnFlyer,
    Discount,
    DiscountTier,
    ActualPrice,
    COUNT(DateWeekNumber) AS WeeksObserved,
    ROUND(AVG(ChainUnits), 0) AS AvgWeeklyUnits,
    ROUND(AVG(ChainSales), 2) AS AvgWeeklySales,
    ROUND(AVG(ChainMargin), 2) AS AvgWeeklyMargin,
    ROUND(AVG(ChainTransactions), 0) AS AvgWeeklyTransactions,
    ROUND(AVG(StoreCount), 0) AS AvgStoresActive,
    ROUND(UnitCost * AVG(ChainUnits), 2) AS TotalCostPerWeek,
    ROUND(AVG(ChainMargin) / AVG(ChainSales) * 100, 1) AS GrossMarginPct,
    ROUND(AVG(ChainMargin) / AVG(ChainUnits), 4) AS MarginPerUnit,
    CASE WHEN AVG(ChainMargin) < 0 THEN 1 ELSE 0 END AS IsLossLeader,
    CURRENT_TIMESTAMP() AS _gold_ts
FROM chain_weekly
WHERE Product = 'Aussie'
GROUP BY Product, UnitCost, PromotionName, OnFlyer, Discount, DiscountTier, ActualPrice
ORDER BY ActualPrice;

-- COMMAND ----------

SELECT ActualPrice, OnFlyer, Discount, WeeksObserved,
       AvgWeeklyUnits, AvgWeeklySales, AvgWeeklyMargin,
       GrossMarginPct, IsLossLeader
FROM gold_loss_leader
ORDER BY ActualPrice
LIMIT 20;

-- COMMAND ----------

SELECT COUNT(*) AS row_count FROM gold_loss_leader;

-- COMMAND ----------
-- MAGIC %md ## 6. Summary

-- COMMAND ----------

SELECT 'price_elasticity' AS gold_table, COUNT(*) AS rows FROM gold_price_elasticity
UNION ALL SELECT 'promotion_uplift', COUNT(*) FROM gold_promotion_uplift
UNION ALL SELECT 'weekly_trend',     COUNT(*) FROM gold_weekly_trend
UNION ALL SELECT 'province_summary', COUNT(*) FROM gold_province_summary
UNION ALL SELECT 'loss_leader',      COUNT(*) FROM gold_loss_leader;

-- COMMAND ----------

-- Q1: Best price for units (per product)
SELECT Product, ActualPrice, OnFlyer, DiscountPct,
       AvgWeeklyUnits, RankByUnits, AvgWeeklyMargin, RankByMargin
FROM gold_price_elasticity
WHERE RankByUnits <= 5
ORDER BY Product, RankByUnits;

-- COMMAND ----------

-- Q2: Best price for margin (per product)
SELECT Product, ActualPrice, OnFlyer, DiscountPct,
       AvgWeeklyMargin, GrossMarginPct, RankByMargin, RankByUnits
FROM gold_price_elasticity
WHERE RankByMargin <= 5
ORDER BY Product, RankByMargin;

-- COMMAND ----------

-- Q4: Cost per unit of each product
SELECT Product, UnitCost
FROM silver_dim_product
ORDER BY Product;

-- COMMAND ----------

-- Q7: On-Flyer vs No-Flyer impact — same discount, compare with/without flyer
SELECT Product, Discount, OnFlyer, PromotionName,
       AvgWeeklyUnits, UnitUpliftPct, AvgWeeklyMargin, MarginUpliftPct
FROM gold_promotion_uplift
WHERE Discount > 0
ORDER BY Product, Discount, OnFlyer;

-- COMMAND ----------

-- Q9/Q10: Aussie loss leader detail
SELECT ActualPrice, OnFlyer, WeeksObserved,
       AvgWeeklyUnits, AvgWeeklySales,
       AvgWeeklyMargin, GrossMarginPct, IsLossLeader
FROM gold_loss_leader
ORDER BY ActualPrice;

-- COMMAND ----------
-- MAGIC %md
-- MAGIC ## ✅ Gold Complete
-- MAGIC
-- MAGIC | Table | Answers |
-- MAGIC |-------|--------|
-- MAGIC | `gold_price_elasticity` | Q1: best price for units, Q2: best price for margin |
-- MAGIC | `gold_promotion_uplift` | Q5: 25% discount impact, Q6: 60% discount, Q7: flyer effect |
-- MAGIC | `gold_weekly_trend`     | Q3: seasonality analysis (rolling avg, WoW change) |
-- MAGIC | `gold_province_summary` | Regional performance, province-level breakdown |
-- MAGIC | `gold_loss_leader`      | Q9: Is Aussie $2.49 effective? Q10: 2-for-$5 scenario |